## Analyzing the Impact of Data Heterogeneity and DataRepair on Fairness in Federated Learning
#### This file contains the code used to generate the results of the paper

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

import tensorflow as tf
tf.config.set_visible_devices([], "GPU")

import random
import csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import ot
from scipy.spatial.distance import cdist
from scipy import stats
from scipy.stats import norm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import tensorflow_federated as tff

Loading the Dataset

In [ ]:
df = cargar_datos('./adult_dataset/adult.data.csv', './adult_dataset/adult.test.csv')

(Exp 1) : i.i.d baseline

In [ ]:
VARIABLE_SENSIBLE = 'Sex'  # 'Sex' u 'OrigEthn' o 'Target' si el repartimos por la target
MODELO = 'simple'
PORCENTAJE_TEST = 0.2
N_REPETICIONES = 50
NUM_CLIENTES = 100
N_ROUNDS = 64
MAX_DATOS = 10000
MODO_SESGO_CLIENTES = 'aleatorio' # 'mayoritario', 'minoritario' o 'aleatorio'

C_MIN = 0.1
C_MAX = 0.1

# Modo de reparto:
#    0: iid
#    1: no-iid por variable sensible S
#    2: no-iid por target Y (mismo esquema que S)
MODO_REPARTO = 0

FICHERO_PROPORCIONES = "proporciones_clientes.csv"

res1 = algoritmo_principal(
    VARIABLE_SENSIBLE,
    MODELO,
    PORCENTAJE_TEST,
    N_REPETICIONES,
    NUM_CLIENTES,
    N_ROUNDS,
    MAX_DATOS,
    MODO_SESGO_CLIENTES,
    C_MIN,
    C_MAX,
    MODO_REPARTO,
    nombre_exp="Exp1"
)

(Exp 2): Non-i.i.d by sensitive attribute, random selection

In [ ]:
VARIABLE_SENSIBLE = 'Sex'  # 'Sex' u 'OrigEthn' o 'Target' si el repartimos por la tarjet
MODELO = 'simple'
PORCENTAJE_TEST = 0.2
N_REPETICIONES = 50
NUM_CLIENTES = 100
N_ROUNDS = 64
MAX_DATOS = 10000
MODO_SESGO_CLIENTES = 'aleatorio' # 'mayoritario', 'minoritario' o 'aleatorio'

C_MIN = 0.1
C_MAX = 0.1

# Modo de reparto:
#    0: iid
#    1: no-iid por variable sensible S
#    2: no-iid por target Y (mismo esquema que S)
MODO_REPARTO = 1

FICHERO_PROPORCIONES = "proporciones_clientes.csv"

res3 = algoritmo_principal(
    VARIABLE_SENSIBLE,
    MODELO,
    PORCENTAJE_TEST,
    N_REPETICIONES,
    NUM_CLIENTES,
    N_ROUNDS,
    MAX_DATOS,
    MODO_SESGO_CLIENTES,
    C_MIN,
    C_MAX,
    MODO_REPARTO,
    nombre_exp="Exp2"
)

(Exp 3): Non-i.i.d by sensitive attribute, majority-based selection (Exp 3

In [ ]:
VARIABLE_SENSIBLE = 'Sex'  # 'Sex' u 'OrigEthn' o 'Target' si el repartimos por la tarjet
MODELO = 'simple'
PORCENTAJE_TEST = 0.2
N_REPETICIONES = 50
NUM_CLIENTES = 100
N_ROUNDS = 64
MAX_DATOS = 10000
MODO_SESGO_CLIENTES = 'mayoritario' # 'mayoritario', 'minoritario' o 'aleatorio'

C_MIN = 0.1
C_MAX = 0.1

# Modo de reparto:
#    0: iid
#    1: no-iid por variable sensible S
#    2: no-iid por target Y (mismo esquema que S)
MODO_REPARTO = 1

FICHERO_PROPORCIONES = "proporciones_clientes.csv"

res3 = algoritmo_principal(
    VARIABLE_SENSIBLE,
    MODELO,
    PORCENTAJE_TEST,
    N_REPETICIONES,
    NUM_CLIENTES,
    N_ROUNDS,
    MAX_DATOS,
    MODO_SESGO_CLIENTES,
    C_MIN,
    C_MAX,
    MODO_REPARTO,
    nombre_exp="Exp3"
)

(Exp 4): Non-i.i.d by sensitive attribute, minority-based selection

In [ ]:
VARIABLE_SENSIBLE = 'Sex'  # 'Sex' u 'OrigEthn' o 'Target' si el repartimos por la tarjet
MODELO = 'simple'
PORCENTAJE_TEST = 0.2
N_REPETICIONES = 50
NUM_CLIENTES = 100
N_ROUNDS = 64
MAX_DATOS = 10000
MODO_SESGO_CLIENTES = 'minoritario' # 'mayoritario', 'minoritario' o 'aleatorio'

C_MIN = 0.1
C_MAX = 0.1

# Modo de reparto:
#    0: iid
#    1: no-iid por variable sensible S
#    2: no-iid por target Y (mismo esquema que S)
MODO_REPARTO = 1

FICHERO_PROPORCIONES = "proporciones_clientes.csv"

res2 = algoritmo_principal(
    VARIABLE_SENSIBLE,
    MODELO,
    PORCENTAJE_TEST,
    N_REPETICIONES,
    NUM_CLIENTES,
    N_ROUNDS,
    MAX_DATOS,
    MODO_SESGO_CLIENTES,
    C_MIN,
    C_MAX,
    MODO_REPARTO,
    nombre_exp="Exp4"
)

(Exp 5): Dirichlet Non-i.i.d, random-based selection

In [ ]:
VARIABLE_SENSIBLE = 'Sex'
MODELO = 'simple'
PORCENTAJE_TEST = 0.2
N_REPETICIONES = 50
NUM_CLIENTES = 100
N_ROUNDS = 64
MAX_DATOS = 10000
MODO_SESGO_CLIENTES = 'aleatorio'

C_MIN = 0.1
C_MAX = 0.1

# Modo de reparto:
#    0: iid
#    1: no-iid por variable sensible S
#    2: no-iid por target Y (linspace)
#    3: no-iid por target Y (Dirichlet)   ← NEW
MODO_REPARTO = 3   # ← change this to 3

BETA = 0.25         # ← NEW: 0.1 extreme, 0.5 moderate, 10.0 near-iid

FICHERO_PROPORCIONES = "proporciones_clientes.csv"

res2 = algoritmo_principal(
    VARIABLE_SENSIBLE,
    MODELO,
    PORCENTAJE_TEST,
    N_REPETICIONES,
    NUM_CLIENTES,
    N_ROUNDS,
    MAX_DATOS,
    MODO_SESGO_CLIENTES,
    C_MIN,
    C_MAX,
    MODO_REPARTO,
    BETA,            # ← NEW
    nombre_exp="Exp_5"
)